<a href="https://colab.research.google.com/github/Mechanics-Mechatronics-and-Robotics/CV-2026/blob/main/Week_06/Lab6_Image_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Lab 6. Image Segmentation.**
Instructor: Alexey Kornaev \\
TA: Kirill Yakovlev


### Acknowledgement
Additional thanks for maintaining this lab by Rauf, Marcus, Rufina, Alex, Batul, Karam and Fahim \\



In [ ]:
!pip install segmentation-models-pytorch
!pip install 'git+https://github.com/facebookresearch/segment-anything.git'

In [ ]:
! wget "http://drive.google.com/uc?export=view&id=1EF4ke1LBRyYM0tfbgiV7c0LPogAs7wz0" -O dog.jpg
! wget "http://drive.google.com/uc?export=view&id=106iRabptomx9nfWbwjo1xYQHhX31QuGg" -O beatles.jpg

Starting from importing necessary libraries...

You might notice that this time we have included the cv2 (Open CV) library in our working space.

**For those who faces CV course 1st time I would recommend to go through deeper reading about this extremely important framework that bases a bunch of real-world solutions in CV area.**

[Official website](https://opencv.org/)

RUS Intro:

[Intro to Open CV. Part 1](https://habr.com/ru/articles/519454/)

[Intro to Open CV. Part 2](https://habr.com/ru/articles/528144/)

[Intro to Open CV. Part 3](https://habr.com/ru/articles/539228/)

[Intro to Open CV. Part 4](https://habr.com/ru/articles/547218/)

Let's start out lab with importing necessary libraries...

In [ ]:
import copy
import os
import random
import shutil
import zipfile
from math import atan2, cos, sin, sqrt, pi, log
from typing import AnyStr, Any, Callable
import cv2, math, time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from numpy import linalg as LA
from torch import optim, nn
from torch.utils.data import DataLoader, random_split
from torch.utils.data.dataset import Dataset
from torchvision import transforms as T
from torchvision import models
from tqdm import tqdm

import segmentation_models_pytorch as smp
from segmentation_models_pytorch.encoders import get_preprocessing_fn

%matplotlib inline
from IPython.display import clear_output
plt.rcParams["figure.figsize"] = (32, 60) # (w, h)

In [ ]:
# Functions for helping
def read_image(filename: str, grayscale: bool = False, fx: float = 1.0, fy: float = 1.0):
    if grayscale:
      img_result = cv2.imread(filename, cv2.IMREAD_GRAYSCALE)
    else:
      imgbgr = cv2.imread(filename, cv2.IMREAD_COLOR)
      # convert to rgb
      img_result = cv2.cvtColor(imgbgr, cv2.COLOR_BGR2RGB)
    # resize
    if fx != 1.0 and fy != 1.0:
      img_result = cv2.resize(img_result, None, fx=fx, fy=fy, interpolation = cv2.INTER_CUBIC)
    return img_result

def show_in_row(list_of_images: list, titles: list = None, disable_ticks: bool = False):
  if type(list_of_images) != list:
    list_of_images = [list_of_images]
  count = len(list_of_images)
  for idx in range(count):
    subplot = plt.subplot(1, count, idx+1)
    if titles is not None:
      subplot.set_title(titles[idx])
    img = list_of_images[idx]
    cmap = 'gray' if (len(img.shape) == 2 or img.shape[2] == 1) else None
    subplot.imshow(img, cmap=cmap)
    if disable_ticks:
      plt.xticks([]), plt.yticks([])
  plt.show()

def show_in_col(list_of_images: list, titles: list = None, disable_ticks: bool = False):
  if type(list_of_images) != list:
    list_of_images = [list_of_images]
  count = len(list_of_images)
  for idx in range(count):
    subplot = plt.subplot(count, 1, idx+1)
    if titles is not None:
      subplot.set_title(titles[idx])

    img = list_of_images[idx]
    cmap = 'gray' if (len(img.shape) == 2 or img.shape[2] == 1) else None
    subplot.imshow(img, cmap=cmap)
    if disable_ticks:
      plt.xticks([]), plt.yticks([])
  plt.show()

The main topic for this lab is **Image Segmentaion** - task about making a model for generating a cohesive masks . But what do we mean cohesive? Well it depends upon our tasks. And there exixts a lot of them. Segmentation is a very broad filed where usually we distinguish:


1.   Instance segmentaion
2.   Edge detection
3.   Super pixelization
4.   Foreground segmetation
5.   Semantic segmentation
6.   Panoptioc segmentation

They can exist interchangeably sometimes or can accomplish a single objective. We will go through some of those.

Let's start with an instance segmentation as a classical approach to  identify and delineate individual instances of objects within an image.

## **Simple Linear Iterative Clustering (SLIC)** - Super pixelization

A superpixel can be defined as a group of pixels that share common characteristics (like pixel intensity).

![alt text](https://ivrl.epfl.ch/wp-content/uploads/2018/08/54082_combo.jpg)



This algorithm is inspired by the k-means algorithm, and its goal is to cluster
image pixels into segments (clusters) based on their color similarity and proximity in the image plane. The principle is pretty straightforward:

1. In the first step, K segment centers are regularly sampled on the image plane and moved to the locations corresponding to the lowest gradient position in a 3 x 3 neighborhood **-** this is done to avoid placing the center at an edge and to reduce the chances of choosing a noisy pixel

2. Afterwards, each image pixel is assigned to one of the segment based on the distance, i.e. the distance from a pixel i to all the segment centers is computed, and the pixel is assigned to the segment whose center has the lowest distance to i **-** we are reducing a search space assuming that pixels that are associated with a segment lie within a 2S x 2S area around the center of this segment on the x,y plane

3. After all the pixels are associated with the nearest cluster center, a new center is computed as the average [L, A, B, x, y] (or [R, G, B, x, y]) vector of all the pixels belonging to the cluster. This process of associating pixels with the nearest cluster center and recomputing the center is iteratively repeated until convergence

The distance measure Ds is defined as the sum of the distance in color space dRGB and the distance in spatial space dxy normalized by the grid interval S:

<img alt="first image" src="https://i.postimg.cc/wBXJ9P4G/Screenshot-6-11-2024-12855-mrl-cs-vsb-cz.jpg" width=500>

### The Algorithm shortly

1) Convert image to LAB colorspace<br>
2) Initalize **K** clusters `[L, A, B, x, y]` in low gradient area<br>
3) For each pixel find closest cluster <br>
4) Move cluster to mean pixel <br>
5) Repeat steps 3 & 4 10 times <br>
6) Connect pixels to clusters by connectivity (?)

----

More information [here](https://jayrambhia.com/blog/superpixels-slic)


First let's check our images. This time we will work with image from your last homework.


In [ ]:
# check image
orig_img = read_image("dog.jpg", False, 0.25, 0.25)
orig_img2 = read_image("beatles.jpg", False, 0.5, 0.5)
show_in_row([orig_img, orig_img2])

Next we need to transform our colorspace from commonly used **RGB** to **LAB** color space.

In [ ]:
def to_lab(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2LAB)

# test lab
lab_img = to_lab(orig_img2)

L, A, B = cv2.split(lab_img)
show_in_col([lab_img, L, A, B], ["LAB", "L", "A (green - red)", "B (blue - yellow)"])

After changing color space, let's initialize cluster centers avoiding object edges...

In [ ]:
def find_local_minimum(gray_img: np.array, pos: tuple, check_grid_area: tuple) -> tuple:

    # return pos
    min_pos = pos

    # Define vertical search window centered around pos
    from_y = (min_pos[0] - int(check_grid_area[0] / 2))
    to_y = from_y + check_grid_area[0]

    # min_val = gray_img[pos[0], pos[1]]

    # Define horizontal search window
    from_x = (min_pos[1] - int(check_grid_area[1] / 2))
    to_x = from_x + check_grid_area[1]

    # Extract local neighborhood
    sub_img = gray_img[from_y:to_y, from_x:to_x]

    # Find index of smallest value (lowest gradient pixel)
    # argmin() finds index of smallest gradient value
    # unravel flat index to 2D coordinates
    offset = np.unravel_index(sub_img.argmin(), sub_img.shape)

    # Converting local offset to global image coordinates
    minimum = (offset[0] + from_y, offset[1] + from_x)

    return minimum


def slic_init(img: np.array, clusters_count: int, check_grid_area: tuple = None) -> list:
    # initalize clusters in low gradient
    # returns list of clusters [{"pos": (x, y), "color": (color_tuple)}]

    # Compute horizontal and vertical gradients
    x = cv2.Sobel(img, -1, 1, 0, borderType=cv2.BORDER_REPLICATE)
    y = cv2.Sobel(img, -1, 0, 1, borderType=cv2.BORDER_REPLICATE)

    # Combine gradients across color channels
    yx_grad = np.abs(np.sum(x, 2)) + np.abs(np.sum(y, 2))

    # show_in_row(x, y, yx_grad)
    # compute grid step size
    step = (img.shape[0] * img.shape[1] / float(clusters_count)) ** 0.5
    step = int(step)
    if check_grid_area is None:
        # Search window is 20% of grid spacing
        check_grid_area = (int(step * 0.2), int(step * 0.2))

    clusters = []
    row_idx = 0
    height, width = img.shape[:2]
    # print(height, width)

    for r in range(step, int(height - check_grid_area[0]), step):
        offset = step if row_idx % 2 == 0 else step/2
        offset = int(offset)
        row_idx += 1

        for c in range(offset, int(width - check_grid_area[1]), step):
    #  Move to low-gradient position avoiding edges
            yx = find_local_minimum(yx_grad, (r, c), check_grid_area)
            color = img[yx[0], yx[1]]
            clusters.append({"pos": yx, "color": color})

    return clusters

def draw_cluster_centers(img, color, clusters):
    for c in clusters:
        pos = c["pos"]
        pos = (int(pos[1]), int(pos[0]))
        cv2.circle(img, pos, 3, color, -1)

n_count = 150
clusters = slic_init(lab_img, n_count)
print("Expected:",n_count, "Result:", len(clusters))

# debug draw
debug_img = orig_img2.copy()
draw_cluster_centers(debug_img, (0, 255, 0), clusters)
show_in_col(debug_img)

So we have found 148 potential cluster centers. Not let's move on onto next the stage trying to find segments by calcualting distance metric between each pixel and a cluster.

In [ ]:
def do_step(lab_img: np.array, clusters: list, m: float = 45) -> list:
    # return list of new clusters

    big_value = 1000000.0
    distances = big_value * np.ones(lab_img.shape[:2])
    clusters_indexes = np.ones(lab_img.shape[:2]) * (-1)

    new_clusters = []
    step = (lab_img.shape[0] * lab_img.shape[1] / float(len(clusters))) ** 0.5

    # m controls shape, small → more color sensitive, large → more spatially compact
    dist_multiplier = m / step

    # each cluster searches only within 2S × 2S region
    max_distance = 2 * step

    for idx, c in enumerate(clusters):
        p = c["pos"]

        # define search area to neighborhood
        from_y = int(max(0, p[0] - max_distance))
        to_y = int(min(lab_img.shape[0], p[0] + max_distance + 1))
        from_x = int(max(0, p[1] - max_distance))
        to_x = int(min(lab_img.shape[1], p[1] + max_distance + 1))

        # compute Euclidean distance in image plane
        yy, xx = np.ogrid[from_y : to_y, from_x : to_x]
        pix_dist = ((yy-p[0])**2 + (xx-p[1])**2)**0.5

        # euclidean color difference
        color_dist = np.abs(lab_img[from_y:to_y, from_x:to_x] - c["color"])
        color_dist = np.sqrt(np.sum(np.square(color_dist), axis=2))

        # SLIC metric
        sum_dist = ((dist_multiplier * pix_dist)**2 + color_dist**2)**0.5

        # assign pixel to closest cluster found so far
        indexes = distances[from_y:to_y, from_x:to_x] > sum_dist
        distances[from_y:to_y, from_x:to_x][indexes] = sum_dist[indexes]
        clusters_indexes[from_y:to_y, from_x:to_x][indexes] = idx

    indnp = np.mgrid[0:lab_img.shape[0], 0:lab_img.shape[1]].transpose(1, 2, 0)
    for k in range(len(clusters)):

        # get mask of assigned pixels
        idx = (clusters_indexes == k)

        # recompute mean color and position
        new_color = np.mean(lab_img[idx], axis=0)
        new_pos = np.mean(indnp[idx], axis=0)
        new_clusters.append({'pos': new_pos, 'color': new_color})
    # show_in_col([clusters_indexes==10, lab_img])
    return new_clusters, clusters_indexes

def color_clusters(img, clusters, clusters_indexes):
  draw_image = img.copy()
  for k in range(len(clusters)):
      idx = (clusters_indexes == k)
      draw_image[idx] = clusters[k]['color']
  return cv2.cvtColor(draw_image, cv2.COLOR_LAB2BGR)

test_clusters = clusters.copy()
debug_img = orig_img2.copy()
draw_cluster_centers(debug_img, (0, 255, 0), test_clusters)
for i in range(10):
    color = (i*5, 0, 255)
    test_clusters, cluster_indices = do_step(lab_img, test_clusters)
    draw_cluster_centers(debug_img, color, test_clusters)
    print("step", i)
show_in_row([debug_img, color_clusters(orig_img2, test_clusters, cluster_indices), cluster_indices])
print("done")

You can do it the same by using built-in method in cv2 or skimage libraries. You can find it as a [superpixels](https://docs.opencv.org/3.4/df/d6c/group__ximgproc__superpixel.html) method, which is a popular name for such algorithms as the SLIC one.

In [ ]:
from cv2.ximgproc import createSuperpixelLSC
from skimage.segmentation import slic

## **U-Net** - Image Segmentation
https://arxiv.org/pdf/1505.04597v1

  <img src="https://i.postimg.cc/N0MPQ5BB/unet.webp" width="900" height="600"/>

The U-Net architecture is characterized by its U-shaped structure, which gives it its name. It consists of an encoding path and a decoding path.

**Encoding Path:** This part of the network captures the context of the input image by using a series of convolutional and max-pooling layers to downsample the spatial dimensions. Basically it is focused on contracting our image.

**Decoding Path:** The decoding path uses upsampling and convolutional layers to produce a segmentation map that has the same spatial dimensions as the input image. It “expands” the contracted images.

Why it is so effective?

1. contracting-expanding principle to preserve "global context" on the first step and localization on the second step

2. skip connections (grey arrows in the figure above) which connect the encoding and decoding paths by merging features. This helps retain spatial details lost during downsampling and helps in preserving the image's local and global context. By maintaining this spatial information, U-Net achieves more accurate segmentation masks. The skip connections assist the network in grasping the relationships between image parts, leading to improved segmentation results.

So let's construct step-by-step, basically we need to write the code for the following structures:

- Downsampling step (encoder)
- Upsampling step (decoder)
- Double-convolution operation

First you might notice the Double Convolution that is repeated in each step (**blue arrow**). As it is shown in the picture, it consist on two convolutions of 3x3 followed by ReLU activation.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv_op = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv_op(x)

Next part is downsampling phase which is basically about double-convolution and max-pooling operation.

In [ ]:
class DownSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        down = self.conv(x)
        p = self.pool(down)

        return down, p

You might notice that in thsi part we save the convolution tensor. As you remember we will need this to make skip connections with our Upsampling phase. This will allow us to fuse global and local features to make a better localization. That convolutioned tensor is later on concatenated with an upsampled tensor with its own dimension.

The upsampling part is done with a deconvolution (green arrow) followed by the double convolution. As we can appreciate, there are four copy and a crop (gray arrow), once before every MaxPooling.

In this case **Upsample** receive two tensors, because Downsample returns two variables *p* and *down* and the latter is saved to be later on concatenated with the output of the Upsample class (i.e., the nn.ConvTranspose2d).

In [ ]:
class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, in_channels//2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        x = torch.cat([x1, x2], 1)
        return self.conv(x)

Finally we need to connect all the steps into one architecture...

In [ ]:
class U_Net(nn.Module):
  def __init__(self, in_channels, num_classes):
    super().__init__()

    self.down_convolution_1 = DownSample(in_channels, 64)
    self.down_convolution_2 = DownSample(64, 128)
    self.down_convolution_3 = DownSample(128, 256)
    self.down_convolution_4 = DownSample(256, 512)

    self.bottle_neck = DoubleConv(512, 1024)

    self.up_convolution_1 = UpSample(1024, 512)
    self.up_convolution_2 = UpSample(512, 256)
    self.up_convolution_3 = UpSample(256, 128)
    self.up_convolution_4 = UpSample(128, 64)

    self.out = nn.Conv2d(in_channels=64, out_channels=num_classes, kernel_size=1)

  def forward(self, x):
      down_1, p1 = self.down_convolution_1(x)
      down_2, p2 = self.down_convolution_2(p1)
      down_3, p3 = self.down_convolution_3(p2)
      down_4, p4 = self.down_convolution_4(p3)

      b = self.bottle_neck(p4)

      up_1 = self.up_convolution_1(b, down_4)
      up_2 = self.up_convolution_2(up_1, down_3)
      up_3 = self.up_convolution_3(up_2, down_2)
      up_4 = self.up_convolution_4(up_3, down_1)

      out = self.out(up_4)
      return out

Let's check that we've arranged everything correctly

In [ ]:
input_image = torch.rand((1,3,512,512))
model = U_Net(3,10)
output = model(input_image)
print(output.size())

Finally, you can train your own model for segmentation. There exists different ways. One way is to use **segmentaion models** library. As it is name suggests you can use pretrained weights to outline your own model. It has pretty understandable logic and based on pytorch framework that you are already familiar with.

[Github repository](https://github.com/qubvel-org/segmentation_models.pytorch/tree/main)

You can find different kind of training like binary and multiclass segmentation.
Usually making dataset for segmentation model training consists of two things: images and masks where a segmentation model should predict the latter as good as possible.

Let's try make a `random` segmentation prediction...

In [ ]:
model = smp.Unet(
    encoder_name="resnet34",        # choose encoder, e.g. mobilenet_v2 or efficientnet-b7
    encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
    in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
    classes=3,                      # model output channels (number of classes in your dataset)
)

# preprocess_input = get_preprocessing_fn('resnet34', pretrained='imagenet')
orig_img = read_image("dog.jpg", False, 0.25, 0.25)
trf = T.Compose([T.ToTensor(),
                  T.Resize(512)],)
input_transformed = trf(orig_img).unsqueeze(0)


model = smp.Unet("resnet34", encoder_weights="imagenet", classes = 5)
model.eval()

with torch.inference_mode():
    output_mask = model(input_transformed)



# input_transformed = trf(orig_img).unsqueeze(0)

mask = torch.nn.functional.interpolate(
    output_mask, size=orig_img.shape[:2], mode="bilinear", align_corners=False
)
mask = mask[0].argmax(0).cpu().numpy()
# Plot results
plt.figure(figsize=(12, 6))

plt.subplot(121)
plt.axis("off")
plt.imshow(orig_img)
plt.title("Input Image")

plt.subplot(122)
plt.axis("off")
plt.imshow(mask)
plt.title("Output Mask")

plt.show()

Oh, looking not so good... I would even say it is somewaht random. Maybe training your own segmentation model in that sense is just inevitable...


## **Semantic Segmentation**

Semantic Segmentation is an image analysis task in which we classify each pixel in the image into a class.

Similar to what us humans do all the time by default, when are looking then whatever we are seeing if we think of that as an image then we know what class each pixel of the image belongs to.

Essentially, Semantic Segmentation is the technique through which we can achieve this in Computers. So, let's say we have the following image.

![](https://i.postimg.cc/y85nDwP6/Screenshot-6-11-2024-11918-www-superannotate-com.jpg)

And then given the above image its semantically segmentated image would be the following

![](https://i.postimg.cc/2S328znX/Screenshot-6-11-2024-11926-www-superannotate-com.jpg)

As you can see, that each pixel in the image is classified to its respective class.This is in most simple terms what Semantic Segmentation is.

## Applications of Semantic Segmentation


The most common use case for the Semantic Segmentation is in:

1. **Autonomous Driving**

  <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSE91Qsc9vk1SAFV8-h0wdu-Uyk7hN9P6S3Qw&s" width="400"/>
  
  In autonomous driving, the image which comes in from the camera is semantically segmented, thus each pixel in the image is classified
  into a class. This helps the computer understand what is present in the its surroundings and thus helps the car act accordingly.


2. **Facial Segmentation**

  <img src="https://drive.google.com/uc?export=view&id=1FiH7QY22g96TYcLIJsdPC1JfAhawh3k2" width="400"/> <br/>
  <small> Source: https://github.com/massimomauro/FASSEG-repository/blob/master/papers/multiclass_face_segmentation_ICIP2015.pdf - does not work atm</small>

  Facial Segmentation is used for segmenting each part of the face into a category, like lips, eyes etc. This technique is used for
  many purposes such as gender estimation, age estimation, facial expression analysis, emotional analysis and more.
  

3. **Indoor Object Segmentation**

  <img src="https://i.postimg.cc/bYCnY1Lp/1-Figure1-1.png" width="400"/><br/>
  <small> Source: http://buildingparser.stanford.edu/dataset.html (oops!) </small>

  Guess where is this used? In AR (Augmented Reality) and VR (Virtual Reality). AR applications when required segments the entire indoor area to understand where there
  are chairs, tables, people, wall, and other obstacles and so on.


4. **Geo-Land Sensing**

  <img src="https://ars.els-cdn.com/content/image/1-s2.0-S0924271616305305-fx1_lrg.jpg" width="400"/> <br/>
  <small> Source: https://www.sciencedirect.com/science/article/pii/S0924271616305305 </small>

  Geo Land Sensing is a way of categorizing each pixel in satellite images into a category such that we can track the land cover of each
  area. So, say in some area there is a heavy deforestation taking place then appropriate measures can be taken.

5. **Interaction between humans and autonomous systems**

  https://github.com/facebookresearch/partnr-planner/tree/main/


### Using pytorch for Semantic Segmentation

Firstly, we need to understand the inputs and outputs of these semantic segmentation models, as they are a bit different with what we have previosuly faced in our labs.<br/>

Usually such models expect a 3-channeled image which is normalized with the Imagenet mean and standard deviation (we talked about that on the previous lab), i.e., <br/>
`mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]`

So, the input is `[Ni x Ci x Hi x Wi]`<br/>
where,
- `Ni` -> the batch size
- `Ci` -> the number of channels (which is 3)
- `Hi` -> the height of the image
- `Wi` -> the width of the image

And the output of the model is `[No x Co x Ho x Wo]`<br/>
where,
- `No` -> is the batch size (same as `Ni`)
- `Co` -> **is the number of classes that the dataset have!**
- `Ho` -> the height of the image (which is the same as `Hi` in almost all cases)
- `Wo` -> the width of the image (which is the same as `Wi` in almost all cases)

However,

The `torchvision` models outputs an `OrderedDict` and not a `torch.Tensor` <br/>
And in `.eval()` mode it just has one key `out` and thus to get the output we need to get the value
stored in that `key`.

The `out` key of this `OrderedDict` is the key that holds the output. <br/>
So, this `out` key's value has the shape of `[No x Co x Ho x Wo]`.

Let's take a model from the segmentation part of **torchvision**. For example, it could be **ResNet101** model. You can check the whole list of model for the semantic segmentation problem. As you might notice it is not that big. Also you can check out the metrics for **COCO dataset 2017** predictions.

  <img src="https://drive.google.com/uc?export=view&id=1Ss3jrg7EcwbKhfFs5CcuGo9ATC5DZJWm" width="600"/>

  Let's import the model...

In [ ]:
fcn = models.segmentation.fcn_resnet101(pretrained=True).eval()

Let's work with previosuly downloaded pictures...

In [ ]:
show_in_row([orig_img])

In [ ]:
trf = T.Compose([T.ToTensor(),
                 T.Resize(256),
                 # You can apply center croping when it is needed
                #  T.CenterCrop(224),
                 T.Normalize(mean = [0.485, 0.456, 0.406],
                             std = [0.229, 0.224, 0.225])])

input_image = trf(orig_img).unsqueeze(0)

Let's see what the above code cell does </br>
`T.Compose` is a function that takes in a `list` in which each element is of `transforms` type and </br>
it returns a object through which we can
pass batches of images and all the required transforms will be applied to the images.

Let's pass all the preprocessed images through the model and get the `out` key.<br/>
As we know, the output of the model is a `OrderedDict` so, we need to take the `out` key from that to get the output of the model.

In [ ]:
# Pass the input through the net
out = fcn(input_image)['out']
print (out.shape)

As we can see, the shape is `[1 x 21 x H x W]` as discussed earlier. So, the model was trained on `21` classes and thus our output have `21` channels!<br/>

Now, what we need to do is make this `21` channeled output into a `2D` image or a `1` channeled image (or mask), where each pixel of that image corresponds to a class!

So, the `2D` image, (of shape `[H x W]`) will have each pixel corresponding to a class label, and thus <br/>
for each `(x, y)` in this `2D` image will correspond to a number between `0 - 20` representing a class.

And how do we get there from this `[1 x 21 x H x W]`?<br/>
We take a max index for each pixel position, which represents the class<br/>

In [ ]:
om = torch.argmax(out.squeeze(), dim=0).detach().cpu().numpy()
print(om.shape)
print(np.unique(om))

In [ ]:
# Define the helper function to return a mask
def decode_segmap(image, nc=21):

  label_colors = np.array([(0, 0, 0),  # 0=background
               # 1=aeroplane, 2=bicycle, 3=bird, 4=boat, 5=bottle
               (128, 0, 0), (0, 128, 0), (128, 128, 0), (0, 0, 128), (128, 0, 128),
               # 6=bus, 7=car, 8=cat, 9=chair, 10=cow
               (0, 128, 128), (128, 128, 128), (64, 0, 0), (192, 0, 0), (64, 128, 0),
               # 11=dining table, 12=dog, 13=horse, 14=motorbike, 15=person
               (192, 128, 0), (64, 0, 128), (192, 0, 128), (64, 128, 128), (192, 128, 128),
               # 16=potted plant, 17=sheep, 18=sofa, 19=train, 20=tv/monitor
               (0, 64, 0), (128, 64, 0), (0, 192, 0), (128, 192, 0), (0, 64, 128)])

  r = np.zeros_like(image).astype(np.uint8)
  g = np.zeros_like(image).astype(np.uint8)
  b = np.zeros_like(image).astype(np.uint8)

  for l in range(0, nc):
    idx = image == l
    r[idx] = label_colors[l, 0]
    g[idx] = label_colors[l, 1]
    b[idx] = label_colors[l, 2]

  rgb = np.stack([r, g, b], axis=2)
  return rgb

In [ ]:
rgb = decode_segmap(om)
plt.imshow(rgb); plt.show()

So we have obtained two blots with the human and the dog.

We also can apply all the operations as one function to segment any pictures with classes **known** by the model. We just include all the steps made previously.

**BUT:** Remember if you have numpy format than applying `ToTensor` method should go first

In [ ]:
def segment(net, path, show_orig=True, dev='cpu', img_type='numpy'):
  first_transform = T.ToTensor()
  second_transform = T.Resize(640)
  img = path
  if img_type == 'PIL':
    img = Image.open(path)
    first_transform = T.Resize(640)
    second_transform = T.ToTensor()

  if show_orig: plt.imshow(img); plt.axis('off'); plt.show()
  # Comment the Resize and CenterCrop for better inference results
  trf = T.Compose([first_transform,
                   #T.CenterCrop(224),
                   second_transform,
                   T.Normalize(mean = [0.485, 0.456, 0.406],
                               std = [0.229, 0.224, 0.225])])
  inp = trf(img).unsqueeze(0).to(dev)
  out = net.to(dev)(inp)['out']
  om = torch.argmax(out.squeeze(), dim=0).detach().cpu().numpy()
  rgb = decode_segmap(om)
  plt.imshow(rgb); plt.axis('off'); plt.show()

In [ ]:
!wget -nv "https://www.learnopencv.com/wp-content/uploads/2021/01/person-segmentation.jpeg" -O persons.png
img = Image.open('./persons.png')
plt.imshow(img); plt.show()

Let's move onto a different architecture like DeepLab, which has gone through several stages of evolution.

[DeepLabV3 theory and atrous convolution.](https://arxiv.org/abs/1706.05587)

In [ ]:
fcn = models.segmentation.deeplabv3_resnet101(pretrained=1).eval()

segment(fcn, path='./persons.png', img_type = 'PIL', show_orig=False)
# segment(fcn, path = orig_img, img_type = 'numpy', show_orig=False)

Let's move to Visual Transformers. We are already familiar with HuggingFace library that contains a considerably bigger number of pretrained models, especially Transformer-based one. Today HF containe everything you need to deal with segmentation tasks. You are probably already familiar with that functionality from the previous Home Task.

Let's import libraries...

In [ ]:
from transformers import (MaskFormerImageProcessor,
                          AutoImageProcessor,
                          MaskFormerForInstanceSegmentation)

def show_masks_on_image(raw_image, masks, scores):
    if len(masks.shape) == 4:
      masks = masks.squeeze()
    if scores.shape[0] == 1:
      scores = scores.squeeze()

    nb_predictions = scores.shape[-1]
    fig, axes = plt.subplots(1, nb_predictions, figsize=(15, 15))

    for i, (mask, score) in enumerate(zip(masks, scores)):
      mask = mask.cpu().detach()
      axes[i].imshow(np.array(raw_image))
      show_mask(mask, axes[i])
      axes[i].title.set_text(f"Mask {i+1}, Score: {score.item():.3f}")
      axes[i].axis("off")
    plt.show()

We have already talked about principle and structure of methods in HuggingFace. All we need is to donwload Image Processor and Model itself.

In [ ]:
processor = AutoImageProcessor.from_pretrained("facebook/maskformer-swin-base-coco")
model = MaskFormerForInstanceSegmentation.from_pretrained("facebook/maskformer-swin-base-coco")

Make all necessary processing and prediction stage...

In [ ]:
inputs = processor(images=orig_img2, return_tensors="pt")
with torch.no_grad():
    outputs = model(**inputs)

predicted_semantic_map = processor.post_process_semantic_segmentation(
    outputs, target_sizes=[orig_img2.shape[:2]]
)[0]

In [ ]:
# show_masks_on_image(orig, masks[0], scores)

num_classes = len(np.unique(predicted_semantic_map))
cmap = plt.cm.get_cmap("hsv", num_classes)

overlay = np.zeros(
    (predicted_semantic_map.shape[0], predicted_semantic_map.shape[1], 4)
)

for i, unique_value in enumerate(np.unique(predicted_semantic_map)):
    overlay[predicted_semantic_map == unique_value, :3] = cmap(i)[:3]
    overlay[predicted_semantic_map == unique_value, 3] = 0.5

fig, ax = plt.subplots(figsize=(12, 15))
ax.imshow(orig_img2)
ax.imshow(overlay, interpolation="nearest", alpha=0.9)
plt.axis("off")

plt.show()

## Panoptic segmentation

Additionally image processors can help to solve **panoptic segmentation** as well.

**Panoptic segmentation** is the task of simultaneously performing both semantic segmentation and instance segmentation, assigning each pixel in an image to a category label and distinguishing between individual objects within those categories.

In [ ]:
import matplotlib.patches as mpatches

predicted_semantic_map = processor.post_process_panoptic_segmentation(
    outputs, target_sizes=[orig_img2.shape[:2]]
)[0]

num_instances = len(predicted_semantic_map["segments_info"])
colors = plt.cm.get_cmap("viridis", num_instances)
overlay = np.zeros((orig_img2.shape[0], orig_img2.shape[1], 4))

for i, info in enumerate(predicted_semantic_map["segments_info"]):
    mask = predicted_semantic_map["segmentation"] == info["id"]
    color = colors(i)
    overlay[mask, :3] = color[:3]  # RGB channels
    overlay[mask, 3] = 0.5  # Alpha channel for transparency

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(orig_img2)
ax.imshow(overlay, interpolation="nearest", alpha=0.9)
plt.axis("off")

handles = [
    mpatches.Patch(
        color=colors(i)[:3], label=f"{model.config.id2label[info['label_id']]}-{i}"
    )
    for i, info in enumerate(predicted_semantic_map["segments_info"])
]
plt.legend(handles=handles, bbox_to_anchor=(1.05, 1), loc="upper left")

plt.show()

## SAM
https://arxiv.org/pdf/2304.02643

The final topic will be about **The Segment Anything Model (SAM)**. It can be considered as a foundation model for image segmentation that can “cut out” any object in any image with minimal user input. It produces high quality object masks from input prompts such as points or boxes, and it can be used to generate masks for all objects on image. It has been trained on a dataset of 11 million images and 1.1 billion masks, and has strong zero-shot performance on a variety of segmentation tasks.

Idea: **The promptable segmentation** creatie a model to return a valid segmentation mask given any prompt as well as for a further zero-shot
transfer to downstream segmentation tasks via prompting.

So basically authors proposed that instead of hard-working fine-tuning big models to make a segmentation solution - what if existing and new segmentation tasks to solve through adaptation via prompt engineering.

<img alt="SAM" src="https://i.postimg.cc/Xq1G78RQ/SAM.png" width=900 height=200>

SAM's architecture follows an encoder-decoder paradigm with a specific feature: it's promptable

- **an image encoder** - an MAE pre-trained large Vision Transformer. It converts the input image into a spatial grid of image features called the image embedding. This is a compact representation (e.g., a 64×64 grid for a 1024×1024 image, with 256-channel feature vectors) that captures high-level visual information.


- **a prompt encoder** - responsible for sparse
(points, boxes, text) and dense masks. It represents
points and boxes by positional encodings summed with
learned embeddings for each prompt type and free-form text
with an off-the-shelf text encoder from CLIP. Dense
prompts (i.e., masks) are embedded using convolutions and summed element-wise with the image embedding. Point and box prompts are turned into a set of sparse embeddings (each is a 256-dim vector, preserving location info), while an input mask (if given) is downsampled into a dense embedding matching the image embedding spatial size.

- **a fast mask decoder** - modification of a Transformer decoder block followed by a dynamic mask prediction head. It is responsible to fuse an information. It takes the image embedding (conditioned by any dense prompt) and iteratively refines a set of output tokens that represent candidate masks. Authors stated that they've used modified decoder block with prompt self-attention and cross-attention in two
directions (prompt-to-image embedding and vice-versa) to
update all embeddings. After running two blocks, it goes up-
sampling of the image embedding and an MLP maps the output
token to a dynamic linear classifier, which then computes
the mask foreground probability at each image location.

There is also a special **IoU token** that decoder produces that predicts the quality of each mask and multiple Mask Tokens that each correspond to a distinct mask hypothesis.










Let's take our picture of dog and segment it on the image.

Start with helper functions provided by authors.

In [ ]:
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(coords, labels, ax, marker_size=375):
    pos_points = coords[labels==1]
    neg_points = coords[labels==0]
    ax.scatter(pos_points[:, 0], pos_points[:, 1], color='green', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)
    ax.scatter(neg_points[:, 0], neg_points[:, 1], color='red', marker='*', s=marker_size, edgecolor='white', linewidth=1.25)

In [ ]:
image = cv2.imread('dog.jpg')
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
plt.axis('on')
plt.show()

Let's downlaod weights...

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
# import sys
# sys.path.append("..")
from segment_anything import sam_model_registry, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

predictor = SamPredictor(sam)

In [ ]:
predictor.set_image(image)

Process the image to produce an image embedding...

In [ ]:
input_point = np.array([[3500, 2250]])
input_label = np.array([1])

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
show_points(input_point, input_label, plt.gca())
plt.axis('on')
plt.show()

By **predict** method the model returns masks, quality predictions for those masks, and low resolution mask logits that can be passed to the next iteration of prediction.

In [ ]:
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

**multimask_output** as **True** will return 3 masks. This setting is intended for ambiguous input prompts, and helps the model disambiguate different objects consistent with the prompt.

In [ ]:
for i, (mask, score) in enumerate(zip(masks, scores)):
    plt.figure(figsize=(10,10))
    plt.imshow(image)
    show_mask(mask, plt.gca())
    show_points(input_point, input_label, plt.gca())
    plt.title(f"Mask {i+1}, Score: {score:.3f}", fontsize=18)
    plt.axis('off')
    plt.show()

Ok, that looks much better...


**That's it for today...**

----

# **Home task**

You have to train your own U-Net model to segment imges from a datset you choose. You can take a ready-to-go dataset with prepared pairs of images and masks or create your own.

This is an example:
https://www.cvat.ai/resources/blog/top-datasets-semantic-segmentation

Requirements:
1. Fully prepared code for training your model using pytorch merthods (no pretrained models)

2. You should present your dataset and explain its segmentaion problem

3. Show results after training your architecture in the form of generated masks. You can use `segment` function we have used during the lab of create by your own.

4. Take randomly 3-5 examples from test set and show image, its `label` mask and predicted mask by the model.

<img alt="SAM" src="https://lh7-us.googleusercontent.com/5HvAHCdds1I4VE0-KPNm2pPuEgg9IVlwLhLGbJJFtcOBgT_EdOq0hrW75csAIOK2KFjkdQusDU66x_aokYiatHheGv73ZfXnqD1Tu6BZCc4xdMS8QDnan1gymDF1iLP9pSUgJEHuqPgpIASSvCr06GI" width=600 height=200>




In [ ]:
# **YOUR CODE**